# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, inspecting, and processing data from the FAIR² Mass Spectrometry Analysis dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(metadata.name + ": " + (metadata.description if metadata.description else ""))
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Published on: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs to understand the dataset structure.

**Note:** Entities in Croissant are referenced by their `@id` fields. We will use `@id` as primary selectors for record sets and fields.

In [ ]:
# List available record sets and their fields (by @id and name) in this dataset.
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    # Sometimes the attribute is singular or nested
    record_sets = [getattr(metadata, 'recordSet', None)] if getattr(metadata, 'recordSet', None) else []

if not record_sets:
    print("No record sets found in the metadata. Attempting to list all FileObjects (tables)...")
    # mlcroissant also exposes tables through the to_json structure
    json_meta = metadata.to_json() if hasattr(metadata, 'to_json') else metadata
    if 'tables' in json_meta:
        record_sets = [tbl['@id'] for tbl in json_meta['tables']]
    else:
        # As fallback, enumerate distributable data files
        record_sets = [dist['@id'] for dist in json_meta.get('distribution', [])]

# If record sets are still empty, show an error.
if not record_sets or all(rs is None for rs in record_sets):
    raise RuntimeError("Could not locate any record set in the croissant schema.")

print(f"Found {len(record_sets)} record set(s).\n")
for i, record_set_id in enumerate(record_sets):
    print(f"Record Set {i+1} @id: {record_set_id}")
    try:
        # Print a sample record from the set
        records = dataset.records(record_set=record_set_id)
        first = next(records)
        print(f"  Fields: {list(first.keys())}")
        print(f'  Example record: {dict(list(first.items())[:3])} ...')
    except Exception as e:
        print("  Could not retrieve records due to:", e)
    print('-'*60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below we extract the main table (record set) using its `@id` identified above and preview its columns. All operations reference columns by their Croissant `@id` in the DataFrame.

In [ ]:
# Choose the main record set for in-depth analysis (here we select the first one for demonstration)
main_record_set = record_sets[0]

# Extract all records into a DataFrame
records_iter = dataset.records(record_set=main_record_set)
records_list = list(records_iter)
df = pd.DataFrame(records_list)

print(f"Loaded {len(df)} records from record set @id: {main_record_set}")
print("Columns (field @id):")
print(df.columns.tolist())

df.head()

## 4. Exploratory Data Analysis (EDA)
We now perform basic data processing using columns referenced by their Croissant field `@id`.

Common EDA steps include filtering numeric columns, normalization, and grouping. Please reference columns by their `@id`. Here is an example using a numeric `@id` such as `'cr:field/coverage_percentage'` if present.

In [ ]:
# Example: Find numeric fields and perform filtering/normalization on one

# Identify potential numeric fields
numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64] or pd.api.types.is_numeric_dtype(df[col])]
if not numeric_candidates:
    # Try to coerce likely numeric fields
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric candidate field @id's: {numeric_candidates}")

# We'll use the first found numeric field for demonstration (you may update to a specific @id as needed)
numeric_field_id = numeric_candidates[0]

# Filter records with field > threshold
threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0
filtered_df = df[df[numeric_field_id] > threshold]

print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean):")
print(filtered_df[[numeric_field_id]].head())

# Normalize selected numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Find a categorical field to group by (e.g., protein name, accession)
categorical_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id]

if categorical_candidates:
    group_field_id = categorical_candidates[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the filtered and normalized numeric field, grouped by the selected category if available.

In [ ]:
# Plot distribution of numeric field
plt.figure(figsize=(8,4))
plt.hist(filtered_df[numeric_field_id], bins=30, alpha=0.7)
plt.title(f'Distribution of {numeric_field_id} (filtered)')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouping is possible, show bar plot of group means
if categorical_candidates:
    n_plot = 10
    top_groups = grouped_df.sort_values(numeric_field_id, ascending=False).head(n_plot)
    plt.figure(figsize=(10,6))
    plt.barh(top_groups[group_field_id], top_groups[numeric_field_id])
    plt.xlabel(f"Mean {numeric_field_id}")
    plt.ylabel(group_field_id)
    plt.title(f"Top {n_plot} {group_field_id} by mean {numeric_field_id}")
    plt.gca().invert_yaxis()
    plt.show()

## 6. Conclusion
We loaded and explored the FAIR² mass spectrometry dataset using `mlcroissant`, referencing all data entities by their `@id` fields. We inspected available record sets, loaded the main table to a DataFrame, selected numeric fields for filtering and normalization, and visualized the results. This workflow can be extended for advanced analytics or machine learning, always referencing columns and entities by their Croissant `@id` for provenance and reproducibility.